# 11.23 Hierarchical RL & options

Options compress long action chains into reusable skills.

An option is a temporally extended action with an internal policy and a termination rule, so planning can reason over skills rather than only primitive moves. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 1119
random.seed(SEED)
np.random.seed(SEED)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    power = 1.0
    for reward in rewards:
        total = total + power * float(reward)
        power = power * gamma
    return total


lesson_rewards = [1, 0, 2]
lesson_gamma = 0.9
lesson_return = discounted_return(lesson_rewards, lesson_gamma)
assert abs(lesson_return - 2.62) < 1e-12


ACTIONS = [0, 1, 2, 3]
ACTION_NAMES = ["up", "right", "down", "left"]
DELTAS = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}


def make_grid_env(name, height, width, start, goal, walls=None, slip=0.0, wind=None, sparse=True, horizon=35, obs_noise=0.0):
    walls = set(walls or [])
    wind = wind or {}
    states = []
    state_index = {}
    for row in range(height):
        for col in range(width):
            cell = (row, col)
            if cell not in walls:
                state_index[cell] = len(states)
                states.append(cell)
    env = {
        "name": name,
        "height": height,
        "width": width,
        "start": start,
        "goal": goal,
        "walls": walls,
        "slip": float(slip),
        "wind": wind,
        "sparse": bool(sparse),
        "horizon": int(horizon),
        "states": states,
        "state_index": state_index,
        "n_states": len(states),
        "n_actions": 4,
        "obs_noise": float(obs_noise),
    }
    return env


def step_cell(env, cell, action, rng):
    actual = int(action)
    if rng.random() < env["slip"]:
        actual = int(rng.integers(0, 4))
    row_delta, col_delta = DELTAS[actual]
    row = cell[0] + row_delta
    col = cell[1] + col_delta
    pushed = env["wind"].get((cell[0], cell[1]), (0, 0))
    row = row + pushed[0]
    col = col + pushed[1]
    candidate = (max(0, min(env["height"] - 1, row)), max(0, min(env["width"] - 1, col)))
    if candidate in env["walls"]:
        candidate = cell
    reward = -0.02
    done = candidate == env["goal"]
    if done:
        reward = 1.0
    if not env["sparse"]:
        distance = abs(candidate[0] - env["goal"][0]) + abs(candidate[1] - env["goal"][1])
        reward = reward - 0.01 * distance
    return candidate, reward, done


def transition_table(env):
    n_states = env["n_states"]
    n_actions = env["n_actions"]
    transitions = np.zeros((n_states, n_actions, n_states), dtype=float)
    rewards = np.zeros((n_states, n_actions, n_states), dtype=float)
    rng = np.random.default_rng(SEED + env["n_states"])
    samples = 80
    for state_id, cell in enumerate(env["states"]):
        for action in range(n_actions):
            for _ in range(samples):
                next_cell, reward, done = step_cell(env, cell, action, rng)
                next_id = env["state_index"][next_cell]
                transitions[state_id, action, next_id] = transitions[state_id, action, next_id] + 1.0
                rewards[state_id, action, next_id] = rewards[state_id, action, next_id] + reward
            total = transitions[state_id, action].sum()
            mask = transitions[state_id, action] > 0
            rewards[state_id, action, mask] = rewards[state_id, action, mask] / transitions[state_id, action, mask]
            transitions[state_id, action] = transitions[state_id, action] / total
    return transitions, rewards


def build_f12_env_ladder():
    ladder = [
        make_grid_env("D1 two-state chain", 1, 2, (0, 0), (0, 1), horizon=6, sparse=False),
        make_grid_env("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), slip=0.15, horizon=10, sparse=False),
        make_grid_env("D3 4x4 gridworld", 4, 4, (0, 0), (3, 3), walls={(1, 1), (2, 1)}, horizon=25, sparse=False),
        make_grid_env("D4 stochastic windy grid", 5, 5, (0, 0), (4, 4), walls={(1, 2), (2, 2)}, slip=0.2, wind={(3, 1): (-1, 0), (3, 2): (-1, 0)}, horizon=35, sparse=False),
        make_grid_env("D5 larger sparse grid", 7, 7, (0, 0), (6, 6), walls={(1, 1), (1, 2), (2, 4), (3, 1), (3, 2), (4, 4), (5, 2)}, slip=0.25, wind={(5, 3): (-1, 0)}, horizon=45, sparse=True),
    ]
    assert len(ladder) == 5
    assert [env["n_states"] for env in ladder] == [2, 3, 14, 23, 42]
    return ladder


def greedy_rollout(env, policy, episodes=20):
    rng = np.random.default_rng(SEED + env["height"] + env["width"])
    returns = []
    paths = []
    for episode in range(episodes):
        cell = env["start"]
        rewards = []
        path = [cell]
        for step in range(env["horizon"]):
            state_id = env["state_index"][cell]
            action = int(policy[state_id])
            cell, reward, done = step_cell(env, cell, action, rng)
            rewards.append(reward)
            path.append(cell)
            if done:
                break
        returns.append(discounted_return(rewards, lesson_gamma))
        paths.append(path)
    return float(np.mean(returns)), paths[-1]


def plot_grid_values(ax, env, values, title):
    grid = np.full((env["height"], env["width"]), np.nan)
    for cell, idx in env["state_index"].items():
        grid[cell] = values[idx]
    image = ax.imshow(grid, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    return image


## The concept, built once: option return and stopping\n\nAn option accumulates discounted internal rewards, then bootstraps after termination:\n$$Q(s,o)=\mathbb{E}\left[\sum_{k=0}^{\tau-1}\gamma^k r_{t+k}+\gamma^\tau V(s_{t+\tau})\right].$$\nFor a 4-step option with final reward 1 and terminal value 0.5, the value is $0.9^3+0.9^4\cdot0.5=1.05705$.

In [ ]:

def option_value_iteration(option_rewards, terminal_value, gamma=0.9):
    total = 0.0
    power = 1.0
    for reward in option_rewards:
        total = total + power * reward
        power = power * gamma
    total = total + power * terminal_value
    return float(total)


option_return = option_value_iteration([0.0, 0.0, 0.0, 1.0], terminal_value=0.5, gamma=0.9)
print("option return", round(option_return, 6))
assert abs(option_return - 1.05705) < 1e-10
assert abs(discounted_return([0.0, 0.0, 0.0, 1.0], 0.9) - 0.729) < 1e-12


The reusable planner constructs hand-coded go-to-goal options and compares them with primitive actions in a Bellman update over temporally extended choices.

In [ ]:

def option_rollout_model(env, start_cell, target_cell, gamma=0.9, max_steps=12):
    cell = start_cell
    rewards = []
    path = [cell]
    for _ in range(max_steps):
        if cell == target_cell:
            break
        row_step = int(np.sign(target_cell[0] - cell[0]))
        col_step = int(np.sign(target_cell[1] - cell[1]))
        if abs(target_cell[0] - cell[0]) >= abs(target_cell[1] - cell[1]):
            action = 2 if row_step > 0 else 0
        else:
            action = 1 if col_step > 0 else 3
        next_cell, reward, done = step_cell(env, cell, action, np.random.default_rng(SEED))
        rewards.append(reward)
        cell = next_cell
        path.append(cell)
        if done or cell == target_cell:
            break
    return cell, rewards, path


def plan_with_options(env, gamma=0.9, sweeps=50):
    subgoals = [env["goal"]]
    if env["n_states"] > 10:
        subgoals.append(env["states"][env["n_states"] // 2])
    values = np.zeros(env["n_states"])
    for _ in range(sweeps):
        next_values = values.copy()
        for cell, state_id in env["state_index"].items():
            candidates = []
            for target in subgoals:
                end_cell, rewards, path = option_rollout_model(env, cell, target, gamma=gamma)
                end_id = env["state_index"][end_cell]
                candidates.append(discounted_return(rewards, gamma) + (gamma ** len(rewards)) * values[end_id])
            next_values[state_id] = max(candidates)
        values = next_values
    policy = np.zeros(env["n_states"], dtype=int)
    for cell, state_id in env["state_index"].items():
        target = env["goal"]
        if abs(target[0] - cell[0]) >= abs(target[1] - cell[1]):
            policy[state_id] = 2 if target[0] > cell[0] else 0
        else:
            policy[state_id] = 1 if target[1] > cell[1] else 3
    return values, policy


## The dataset ladder

In [ ]:

ladder = build_f12_env_ladder()
for env in ladder:
    sample_states = env["states"][:5]
    print(env["name"], "states", env["n_states"], "actions", env["n_actions"], "sample", sample_states)


In [ ]:

results = []
artifacts = []
for env in ladder:
    values, policy = plan_with_options(env)
    mean_return, path = greedy_rollout(env, policy)
    results.append({"rung": env["name"], "return": mean_return})
    artifacts.append((env, values, path))
print("rung | return")
for row in results:
    print(row["rung"], round(row["return"], 3))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, (env, values, path) in enumerate(artifacts):
    plot_grid_values(axes[0, col], env, values, env["name"])
axes[1, 0].plot([idx + 1 for idx in range(len(results))], [row["return"] for row in results], marker="o")
axes[1, 0].set_title("return vs option rung")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].set_ylabel("return")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: confusing option length reward with return

Rewarding long options as if length itself were value can prefer wandering. Discounted option returns fix the objective.

In [ ]:

d5 = ladder[-1]
long_option_score = len(option_rollout_model(d5, d5["start"], d5["goal"], max_steps=20)[1])
correct_values, correct_policy = plan_with_options(d5)
correct_return, _ = greedy_rollout(d5, correct_policy)
print("wrong length score", long_option_score)
print("fixed discounted option return", round(correct_return, 3))


## Evaluate it + Practice

- Metric: compare D1-D5 return against a no-skill baseline.
- Sanity check: D1 should solve by hand and match the exact assertion in the build-once cell.
- Ablation: turn off the key idea and verify the metric worsens on D4 or D5.
- Failure signal: curves that look good only because support, entropy, shape, or rollout bias was hidden.

Practice 1: change one ladder parameter and predict the metric direction before running.


Practice 2: add a bottleneck subgoal and compare value heatmaps with and without it.